<a href="https://colab.research.google.com/github/soaresgaab/TestePraticoProjedata/blob/main/Mestrado_UFMA_APE_Microdados_2021_a_2024_(Script_para_Limpeza_dos_dados).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [33]:
import pandas as pd
import numpy as np

from google.colab import drive
drive.mount('/content/drive')

pd.set_option('display.max_columns', None)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [34]:
ANO_ANALISE = 2020

CAMINHO_BASE = '/content/drive/MyDrive/Mestrado UFMA/Análiticas de Aprendizagem/Dados Censo Superior/'

arquivo_cursos = f'{CAMINHO_BASE}MICRODADOS_CADASTRO_CURSOS_{ANO_ANALISE}.CSV'
arquivo_ies = f'{CAMINHO_BASE}MICRODADOS_CADASTRO_IES_{ANO_ANALISE}.CSV'
arquivo_saida = f'{CAMINHO_BASE}df_computacao_limpo_{ANO_ANALISE}.csv'

In [35]:
colunas_cursos = ['NO_CINE_AREA_GERAL', 'NO_CURSO', 'NO_REGIAO', 'SG_UF', 'CO_IES', 'QT_ING', 'QT_ING_FEM', 'QT_MAT', 'QT_MAT_FEM', 'QT_CONC', 'QT_CONC_FEM', 'TP_MODALIDADE_ENSINO']
df_cursos = pd.read_csv(arquivo_cursos, sep=';', encoding='latin1', usecols=colunas_cursos, low_memory=False)
df_ies = pd.read_csv(arquivo_ies, sep=';', encoding='latin1', usecols=['CO_IES', 'TP_CATEGORIA_ADMINISTRATIVA'], low_memory=False)

In [36]:
df_comp = df_cursos[df_cursos['NO_CINE_AREA_GERAL'] == 'Computação e Tecnologias da Informação e Comunicação (TIC)'].copy()

for col in ['NO_CURSO', 'NO_REGIAO', 'SG_UF']:
    df_comp[col] = df_comp[col].astype(str).str.strip().str.upper()

df_comp = df_comp[(df_comp['NO_REGIAO'] != 'NAN') & (df_comp['SG_UF'] != 'NAN')].copy()
df_comp.dropna(subset=['NO_REGIAO', 'SG_UF'], inplace=True)


vars_quant = ['QT_ING', 'QT_ING_FEM', 'QT_MAT', 'QT_MAT_FEM', 'QT_CONC', 'QT_CONC_FEM']
for col in vars_quant:
    df_comp[col] = pd.to_numeric(df_comp[col], errors='coerce').fillna(0)

df_comp = df_comp[(df_comp['QT_ING'] > 0) | (df_comp['QT_MAT'] > 0) | (df_comp['QT_CONC'] > 0)].copy()

In [37]:
df_comp['PERC_ING_FEM'] = np.where(df_comp['QT_ING'] > 0, (df_comp['QT_ING_FEM'] / df_comp['QT_ING']) * 100, 0)
df_comp['PERC_MAT_FEM'] = np.where(df_comp['QT_MAT'] > 0, (df_comp['QT_MAT_FEM'] / df_comp['QT_MAT']) * 100, 0)
df_comp['PERC_CONC_FEM'] = np.where(df_comp['QT_CONC'] > 0, (df_comp['QT_CONC_FEM'] / df_comp['QT_CONC']) * 100, 0)

In [38]:
df_final = pd.merge(df_comp, df_ies, on='CO_IES', how='left')

df_final['TIPO_INSTITUICAO'] = np.where(df_final['TP_CATEGORIA_ADMINISTRATIVA'].isin([1, 2, 3]), 'Pública', 'Privada')
df_final['ANO_CENSO'] = ANO_ANALISE

df_final.to_csv(arquivo_saida, index=False, encoding='utf-8-sig')

In [39]:
total_geral_brasil = len(df_cursos)
df_so_computacao = df_cursos[df_cursos['NO_CINE_AREA_GERAL'] == 'Computação e Tecnologias da Informação e Comunicação (TIC)']

linhas_computacao_bruto = len(df_so_computacao)
linhas_computacao_limpo = len(df_final)

descartados_limpeza = linhas_computacao_bruto - linhas_computacao_limpo
print(f"=== RESUMO DA LIMPEZA DO ANO {ANO_ANALISE} ===")
print(f"Total de cursos de Computação originais: {linhas_computacao_bruto}")
print(f"Total de cursos válidos (após limpeza): {linhas_computacao_limpo}")
print(f"Descartados (nulos ou sem alunos): {descartados_limpeza}")

=== RESUMO DA LIMPEZA DO ANO 2020 ===
Total de cursos de Computação originais: 26529
Total de cursos válidos (após limpeza): 24176
Descartados (nulos ou sem alunos): 2353
